# 5.5 Patient Subtyping & Supervised Clustering

This notebook aims to identify distinct patient subtypes based on the VAE latent representations.
It performs clustering experiments using different feature sets and algorithms, evaluating alignment with known labels (PD vs Control).

## Experiments
1. **Feature Sets**:
   - **Set A (All Active)**: All dimensions with significant information content (KL > 0.05).
   - **Set B (Decorrelated)**: Set A excluding dimensions strongly correlated with Age, Sex, or Scanner (confounders).
2. **Algorithms**:
   - **K-Means**: Hard clustering, assumes spherical clusters.
   - **GMM**: Probabilistic clustering, allows elliptical clusters (Full Covariance).
3. **Evaluation**:
   - **Adjusted Rand Index (ARI)**: Agreement with PD/Control labels (Supervised Check).
   - **Silhouette Score**: Internal cluster separation quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score, homogeneity_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: '%.4f' % x)

## 1. Data Loading & Feature Selection

In [ ]:
# --- PARAMETERS ---
LATENT_CSV = "output/final_train_combined_vae_data.csv"
CORR_SUMMARY_CSV = "output/strict_correlation_analysis_summary.csv"
KL_CSV_PATH = "output/Experiments/BetaScanVAE/beta_scan_results/beta_3.00e-05/kl_divergence.csv"
KL_THRESHOLD = 0.05

# Load Main Data
if os.path.exists(LATENT_CSV):
    df_latent = pd.read_csv(LATENT_CSV)
    print(f"Loaded Latent Data: {df_latent.shape}")
else:
    raise FileNotFoundError(f"Latent data not found at {LATENT_CSV}")

# 1. Define Set A (All Active)
DIM_COLS = [c for c in df_latent.columns if c.startswith('latent_')]
active_dims = []

if os.path.exists(KL_CSV_PATH):
    df_kl = pd.read_csv(KL_CSV_PATH)
    active_dims = df_kl[df_kl['mean_kl'] > KL_THRESHOLD]['dimension'].tolist()
else:
    # Fallback to Variance
    variances = df_latent[DIM_COLS].var()
    active_dims = variances[variances > KL_THRESHOLD].index.tolist()

SET_A = sorted(active_dims, key=lambda x: int(x.split('_')[1]))
print(f"\nFeature Set A (All Active): {len(SET_A)} dimensions")

# 2. Define Set B (Decorrelated)
# Exclude dimensions found significant for Age, Sex, Scanner in 5.1.1
exclude_dims = set()

if os.path.exists(CORR_SUMMARY_CSV):
    df_corr = pd.read_csv(CORR_SUMMARY_CSV)
    # Filter for Confounders (Exclude SBR as that's a disease marker we might WANT to cluster by)
    confounders = ['Age', 'Sex', 'Scanner', 'Manufacturer']
    
    significant_confounders = df_corr[
        (df_corr['Factor'].isin(confounders)) & 
        (df_corr['Survives_Bonferroni'] == True)
    ]
    exclude_dims = set(significant_confounders['Top_Dimension'].unique())
    print(f"Excluding {len(exclude_dims)} dimensions correlated with Demographics/Scanner.")
else:
    print("Warning: Correlation summary not found. Set B will be identical to Set A.")

SET_B = [d for d in SET_A if d not in exclude_dims]
print(f"Feature Set B (Decorrelated): {len(SET_B)} dimensions")

## 2. Clustering Experiment Loop

In [ ]:
def run_clustering_experiments(data, feature_sets, k_range=[2,3,4,5,6]):
    results = []
    
    # Prepare Labels for ARI
    # We need to map strings (PD, Control) to integers if possible, or just pass as is
    # Dropping NaNs in labels for valid comparison
    data_clean = data.dropna(subset=['label']).copy()
    true_labels = data_clean['label']
    
    for set_name, features in feature_sets.items():
        print(f"\nRunning Experiments for {set_name} ({len(features)} dim)...")
        
        X = data_clean[features].values
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        for k in k_range:
            # 1. K-Means
            km = KMeans(n_clusters=k, random_state=42)
            km_labels = km.fit_predict(X_scaled)
            
            res_km = {
                'Feature_Set': set_name,
                'Algorithm': 'KMeans',
                'k': k,
                'ARI': adjusted_rand_score(true_labels, km_labels),
                'Silhouette': silhouette_score(X_scaled, km_labels)
            }
            results.append(res_km)
            
            # 2. GMM
            gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
            gmm_labels = gmm.fit_predict(X_scaled)
            
            res_gmm = {
                'Feature_Set': set_name,
                'Algorithm': 'GMM',
                'k': k,
                'ARI': adjusted_rand_score(true_labels, gmm_labels),
                'Silhouette': silhouette_score(X_scaled, gmm_labels)
            }
            results.append(res_gmm)
            
    return pd.DataFrame(results)

feature_dict = {
    'Set_A_All_Active': SET_A,
    'Set_B_Decorrelated': SET_B
}

df_results = run_clustering_experiments(df_latent, feature_dict)

print("\n--- Clustering Results Summary ---")
print(df_results.sort_values(by='ARI', ascending=False).head(10))

## 3. Visualize Best Clustering Result
We use PCA to project the high-dimensional latent space to 2D and color by the best cluster assignments.

In [ ]:
# Select Best Model based on ARI (Agreement with PD/Control)
best_run = df_results.sort_values(by='ARI', ascending=False).iloc[0]
print(f"\nBest Model: {best_run['Algorithm']} (k={best_run['k']}) on {best_run['Feature_Set']}")
print(f"ARI: {best_run['ARI']:.4f}")

# Re-run prediction to get labels for plotting
features = feature_dict[best_run['Feature_Set']]
X = df_latent.dropna(subset=['label'])[features].values
X_scaled = StandardScaler().fit_transform(X)

if best_run['Algorithm'] == 'KMeans':
    model = KMeans(n_clusters=best_run['k'], random_state=42)
else:
    model = GaussianMixture(n_components=best_run['k'], covariance_type='full', random_state=42)

cluster_labels = model.fit_predict(X_scaled)
true_labels = df_latent.dropna(subset=['label'])['label'].values

# PCA for Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))

# Plot 1: True Labels
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=true_labels, palette='viridis', alpha=0.6)
plt.title("True Labels (PD vs Control)")
plt.xlabel("PC1")
plt.ylabel("PC2")

# Plot 2: Cluster Labels
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=cluster_labels, palette='tab10', alpha=0.6)
plt.title(f"Cluster Labels ({best_run['Algorithm']} k={best_run['k']})")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.tight_layout()
plt.savefig("output/clustering_best_model_pca.png")
plt.show()
print("Saved visualization to output/clustering_best_model_pca.png")

## 4. Save Results

In [ ]:
output_path = "output/patient_clustering_results.csv"
df_results.to_csv(output_path, index=False)
print(f"Clustering Metrics saved to: {output_path}")